In [1]:
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import os

In [2]:
DATA_DIR   = "./data"
MODELS_DIR = "./models"
GENRE_EMBEDDINGS_PATH = f"{DATA_DIR}/embeddings/genre_only_embeddings.json"

In [3]:
# ── Config ─────────────────────────────────────────────────────────────────────
EMBED_DIM  = 128
HIDDEN_DIM = 64
LATENT_DIM = 16
EPOCHS     = 200
BATCH_SIZE = 128
LR         = 1e-3
BETA       = 0.005
PATIENCE   = 30
MIN_DELTA  = 0.0001

In [4]:
# ── Load genre embeddings ──────────────────────────────────────────────────────
with open(GENRE_EMBEDDINGS_PATH) as f:
    raw = json.load(f)

genres    = list(raw.keys())               # ["g_pop rap", "g_jazz", ...]
token2idx = {g: i for i, g in enumerate(genres)}
X         = torch.tensor(list(raw.values()), dtype=torch.float32)

print(f"{len(genres)} genres | embed_dim={X.shape[1]}")

dataset    = TensorDataset(X)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

1493 genres | embed_dim=128


In [5]:
# ── VAE ────────────────────────────────────────────────────────────────────────
class VAE(nn.Module):
    def __init__(self, embed_dim, hidden_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU()
        )
        self.mu_layer     = nn.Linear(hidden_dim, latent_dim)
        self.logvar_layer = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.mu_layer(h), self.logvar_layer(h)

    def reparameterize(self, mu, logvar):
        logvar = torch.clamp(logvar, min=-4, max=4)
        if self.training:
            return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return mu

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z          = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(recon, x, mu, logvar, beta=BETA):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='mean')
    kl_loss    = torch.mean(torch.clamp(
        -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()), min=0.01
    ))
    return recon_loss + beta * kl_loss, recon_loss, kl_loss

In [6]:
# ── Train ──────────────────────────────────────────────────────────────────────
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

model     = VAE(EMBED_DIM, HIDDEN_DIM, LATENT_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_loss, patience_counter = float('inf'), 0
os.makedirs(MODELS_DIR, exist_ok=True)

for epoch in range(EPOCHS):
    model.train()
    total_loss = total_recon = total_kl = 0.0

    for (batch,) in dataloader:
        batch             = batch.to(device)
        recon, mu, logvar = model(batch)
        loss, recon_loss, kl_loss = vae_loss(recon, batch, mu, logvar)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss  += loss.item()
        total_recon += recon_loss.item()
        total_kl    += kl_loss.item()

    n        = len(dataloader)
    avg_loss = total_loss / n
    print(f"Epoch {epoch+1}/{EPOCHS} — loss={avg_loss:.4f} | recon={total_recon/n:.4f} | kl={total_kl/n:.4f}")

    if avg_loss < best_loss - MIN_DELTA:
        best_loss, patience_counter = avg_loss, 0
        torch.save(model.state_dict(), f"{MODELS_DIR}/vae_model.pt")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

Using: cuda
Epoch 1/200 — loss=0.5146 | recon=0.5146 | kl=0.0144
Epoch 2/200 — loss=0.4920 | recon=0.4918 | kl=0.0348
Epoch 3/200 — loss=0.4791 | recon=0.4787 | kl=0.0851
Epoch 4/200 — loss=0.4709 | recon=0.4698 | kl=0.2264
Epoch 5/200 — loss=0.4614 | recon=0.4589 | kl=0.5006
Epoch 6/200 — loss=0.4520 | recon=0.4481 | kl=0.7718
Epoch 7/200 — loss=0.4414 | recon=0.4374 | kl=0.8055
Epoch 8/200 — loss=0.4329 | recon=0.4283 | kl=0.9324
Epoch 9/200 — loss=0.4245 | recon=0.4186 | kl=1.1930
Epoch 10/200 — loss=0.4147 | recon=0.4083 | kl=1.2801
Epoch 11/200 — loss=0.4070 | recon=0.4004 | kl=1.3215
Epoch 12/200 — loss=0.4014 | recon=0.3942 | kl=1.4238
Epoch 13/200 — loss=0.3952 | recon=0.3877 | kl=1.4967
Epoch 14/200 — loss=0.3888 | recon=0.3809 | kl=1.5811
Epoch 15/200 — loss=0.3858 | recon=0.3778 | kl=1.6045
Epoch 16/200 — loss=0.3787 | recon=0.3706 | kl=1.6239
Epoch 17/200 — loss=0.3743 | recon=0.3658 | kl=1.7107
Epoch 18/200 — loss=0.3686 | recon=0.3597 | kl=1.7801
Epoch 19/200 — loss=0.365

In [7]:
# ── Extract latent vectors ─────────────────────────────────────────────────────
model.load_state_dict(torch.load(f"{MODELS_DIR}/vae_model.pt"))
model.eval()

with torch.no_grad():
    mu, _ = model.encode(X.to(device))
    Z     = mu.cpu().numpy()

print(f"Latent vectors: {Z.shape}")  # (1493, 16)

Latent vectors: (1493, 16)


In [8]:
# ── Save latent vectors ────────────────────────────────────────────────────────
latent_embeddings = {genres[i]: Z[i].tolist() for i in range(len(genres))}

with open(f"{DATA_DIR}/embeddings/latent_embeddings.json", "w") as f:
    json.dump(latent_embeddings, f)

print("Saved latent embeddings.")

Saved latent embeddings.
